# V0 batch — all 68 dev doublets

Runs the V0 pipeline (one-vs-rest, σ_cut = 5 px, depth = 0.99, α = 10) on each approved doublet using a uniform 2048×2048 px crop centred on the bookmark cell. Produces:

- `out/<benchmark_id>.png` — 1×4 panel per doublet (inputs, cut, control mask, treated mask).
- `out/_summary.csv` — n_focal control vs treated for every doublet (written incrementally).
- `out/_scorecard.png` — overall split-rate scorecard.
- `out/_heatmap.png` — at-a-glance heatmap, rows=doublets, columns=[control, treated].

**Compute:** 68 × 2 Cellpose-SAM calls on Apple Silicon MPS ≈ 30–60 minutes.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, time, traceback
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd, tifffile, matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
import lib

CROP_PX = 2048
OUT = Path("out"); OUT.mkdir(exist_ok=True)
print(f"V0 batch — α={lib.ALPHA}  σ_diff={lib.SIGMA_DIFFUSE_PX:.2f}px  K_sharp={lib.K_SHARP}  σ_cut={lib.SIGMA_CUT_PX}px  depth={lib.DEPTH_MAX}")
print(f"out: {OUT.resolve()}")

In [ ]:
# Load bench + WSI metadata, pin down which doublets to process
bench = pd.read_parquet(lib.DATA / "benchmark_doublets.parquet")
if "approved" in bench.columns:
    bench = bench[bench["approved"].fillna(False)].copy()
bench = bench.sort_values("benchmark_id").reset_index(drop=True)

with tifffile.TiffFile(lib.DAPI_TIF) as tf:
    WSI_H, WSI_W = tf.series[0].shape[-2:]

wsi_mask_path = lib.DATA / "cpsam_whole_slide" / "masks.tif"
have_wsi_mask = wsi_mask_path.exists()
print(f"{len(bench)} approved doublets")
print(f"WSI {WSI_H}×{WSI_W} | WSI mask available: {have_wsi_mask}")

# Cache static inputs once (gene→lineage, percentiles)
gene_lin = lib.load_lineage_label_map()
percentiles = lib.load_wsi_percentiles()
print(f"gene panel: lineage-mapped {(gene_lin >= 0).sum()} / {len(gene_lin)}")

In [ ]:
# Reusable helpers — local to the batch notebook
def label_overlay(mask, alpha=0.45):
    u = np.unique(mask); u = u[u != 0]
    cm = plt.get_cmap("tab20")(np.linspace(0, 1, max(len(u), 1)))
    out = np.zeros((*mask.shape, 4), dtype=np.float32)
    for i, L in enumerate(u): out[mask == L] = (*cm[i % len(cm)][:3], alpha)
    return out

def count_focal(m, focal):
    a = focal.sum()
    if a == 0: return 0, []
    lbls = np.unique(m[focal]); lbls = lbls[lbls != 0]
    covers = sorted([(int(L), int(((m==L)&focal).sum())/a) for L in lbls], key=lambda kv:-kv[1])
    return sum(1 for _, c in covers if c >= 0.05), covers[:5]

def process_one(row, wsi_mask=None):
    BID = row["benchmark_id"]
    cx_um, cy_um = float(row["x_um"]), float(row["y_um"])
    cps_focal = int(row["cps_id_at_bookmark"])
    pair = row["lineage_pair"]
    y0, y1, x0, x1 = lib.make_roi_bbox_centred(cx_um, cy_um, CROP_PX, WSI_H, WSI_W)

    dapi, s18 = lib.load_morphology(y0, y1, x0, x1)
    py, px, li = lib.load_anchors_in_bbox(y0, y1, x0, x1, gene_lin)
    H, W = s18.shape
    pi_abst, confidence, _ = lib.lineage_posterior(py, px, li, H, W)
    edge_total, top_idx, _ = lib.edge_global_field(pi_abst)
    evidence = lib.cell_evidence(dapi, s18, percentiles)
    cut, _ = lib.cut_field_from_edge(edge_total, confidence, evidence)
    s_cut = lib.apply_cut(s18, cut)

    masks_ctrl, info_ctrl = lib.run_cpsam(dapi, s18)
    masks_cut,  info_cut  = lib.run_cpsam(dapi, s_cut)

    # Focal evaluation (uses WSI mask only as reference for the bookmark cell footprint;
    # this never feeds into the cut field)
    focal = None; n_ctrl = n_cut_ = -1
    cov_ctrl = cov_cut = []
    if wsi_mask is not None:
        focal = (wsi_mask[y0:y1, x0:x1] == cps_focal)
        if focal.any():
            n_ctrl, cov_ctrl = count_focal(masks_ctrl, focal)
            n_cut_, cov_cut = count_focal(masks_cut,  focal)

    # Per-doublet figure
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.2))
    axes[0].imshow(np.log1p(s18), cmap="gray")
    colors = plt.get_cmap("tab10").colors
    for k, L in enumerate(lib.LINEAGES):
        m = li == k
        if m.any(): axes[0].scatter(px[m], py[m], s=2, color=colors[k], alpha=0.7)
    if focal is not None and focal.any():
        axes[0].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.9)
    axes[0].set_title(f"{BID}\n{pair} | 18S + anchors")
    axes[1].imshow(cut, cmap="hot", vmin=0, vmax=1); axes[1].set_title(f"cut field (σ={lib.SIGMA_CUT_PX}, d={lib.DEPTH_MAX})")
    axes[2].imshow(np.log1p(s18), cmap="gray")
    axes[2].imshow(label_overlay(masks_ctrl))
    axes[2].contour(find_boundaries(masks_ctrl, mode="inner").astype(int), levels=[0.5], colors="white", linewidths=0.4)
    if focal is not None and focal.any():
        axes[2].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.9)
    cov_ctrl_str = ' '.join(f'{c*100:.0f}%' for _, c in cov_ctrl[:3])
    axes[2].set_title(f"Control CP-SAM (raw 18S)\nfocal n={n_ctrl}  {cov_ctrl_str}")
    axes[3].imshow(np.log1p(s_cut), cmap="gray")
    axes[3].imshow(label_overlay(masks_cut))
    axes[3].contour(find_boundaries(masks_cut, mode="inner").astype(int), levels=[0.5], colors="white", linewidths=0.4)
    if focal is not None and focal.any():
        axes[3].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.9)
    cov_cut_str = ' '.join(f'{c*100:.0f}%' for _, c in cov_cut[:3])
    delta = "+" if n_cut_ > n_ctrl else ("=" if n_cut_ == n_ctrl else "−")
    axes[3].set_title(f"Treated CP-SAM (s_cut)  [{delta}]\nfocal n={n_cut_}  {cov_cut_str}")
    for a in axes: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    fig.savefig(OUT / f"{BID}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)

    return dict(
        benchmark_id=BID, lineage_pair=pair,
        n_control=n_ctrl, n_treated=n_cut_,
        n_cells_ctrl=info_ctrl["n_cells"], n_cells_cut=info_cut["n_cells"],
        top1_ctrl_pct=cov_ctrl[0][1]*100 if cov_ctrl else 0.0,
        top1_cut_pct =cov_cut[0][1] *100 if cov_cut  else 0.0,
    )

In [ ]:
# Batch loop — checkpoint after each doublet
wsi_mask = None
if have_wsi_mask:
    print("loading WSI mask (3.5 GB) for focal-region eval…")
    wsi_mask = tifffile.imread(wsi_mask_path)
    print(f"  shape={wsi_mask.shape} dtype={wsi_mask.dtype}")

rows = []
t_all = time.time()
for i, row in bench.iterrows():
    t = time.time()
    try:
        r = process_one(row, wsi_mask=wsi_mask)
        rows.append(r)
        delta = "+" if r["n_treated"] > r["n_control"] else ("=" if r["n_treated"] == r["n_control"] else "−")
        print(f"[{i+1:2d}/{len(bench)}] {row['benchmark_id']:<16} ctrl={r['n_control']} cut={r['n_treated']} [{delta}]  ({time.time()-t:.0f}s)")
        pd.DataFrame(rows).to_csv(OUT / "_summary.csv", index=False)
    except Exception as e:
        print(f"[{i+1:2d}/{len(bench)}] {row['benchmark_id']} FAILED — {e}")
        traceback.print_exc()

print(f"\nTotal batch time: {(time.time()-t_all)/60:.1f} min")

In [ ]:
df = pd.read_csv(OUT / "_summary.csv")
n = len(df)
n_split_ctrl = (df["n_control"] >= 2).sum()
n_split_cut  = (df["n_treated"] >= 2).sum()
n_up   = (df["n_treated"] >  df["n_control"]).sum()
n_dn   = (df["n_treated"] <  df["n_control"]).sum()
n_same = n - n_up - n_dn
print("=== V0 scorecard ===")
print(f"  doublets processed: {n}")
print(f"  split-rate  control:  {n_split_ctrl}/{n}")
print(f"  split-rate  treated:  {n_split_cut}/{n}")
print(f"  per-doublet:  improved {n_up}   regressed {n_dn}   same {n_same}")
if n_dn > 0:
    print(f"\n  regressions:")
    for _, r in df[df['n_treated'] < df['n_control']].iterrows():
        print(f"    {r['benchmark_id']}: ctrl={r['n_control']} → cut={r['n_treated']}")

In [ ]:
# Scorecard bar plot
fig, ax = plt.subplots(figsize=(7.5, 4.5))
cats = ["split-rate\ncontrol", "split-rate\ntreated", "improved\n(treated > ctrl)", "regressed\n(treated < ctrl)", "same"]
vals = [n_split_ctrl, n_split_cut, n_up, n_dn, n_same]
colors = ["#888", "#1976d2", "#2e7d32", "#c62828", "#999"]
bars = ax.bar(cats, vals, color=colors)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v}/{n}", ha="center", fontsize=10)
ax.set_ylim(0, n+5)
ax.set_title(f"V0 — boundary-prior 18S cropping on {n} dev doublets")
ax.set_ylabel("n doublets")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); fig.savefig(OUT / "_scorecard.png", dpi=140, bbox_inches="tight"); plt.show()

In [ ]:
# Heatmap: rows = doublets (sorted), cols = [control, treated], cells = n_focal
df_s = df.sort_values("benchmark_id").reset_index(drop=True)
M = df_s[["n_control", "n_treated"]].to_numpy().astype(float)

fig, ax = plt.subplots(figsize=(4.2, max(8, len(df_s) * 0.22)))
im = ax.imshow(M, cmap="YlOrRd", vmin=0, vmax=4, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["control", "treated"], fontsize=10)
ax.set_yticks(range(len(df_s)))
ax.set_yticklabels(df_s["benchmark_id"].str.replace("DB_top100_", "").to_list(), fontsize=7)
for i in range(len(df_s)):
    for j in range(2):
        v = int(M[i, j]) if M[i, j] >= 0 else "–"
        c = "white" if isinstance(v, int) and v >= 3 else "black"
        ax.text(j, i, str(v), ha="center", va="center", color=c, fontsize=7)
# highlight rows where treated > control (improvements)
for i, r in df_s.iterrows():
    if r["n_treated"] > r["n_control"]:
        ax.axhspan(i-0.5, i+0.5, fill=False, edgecolor="green", linewidth=1.4)
    elif r["n_treated"] < r["n_control"]:
        ax.axhspan(i-0.5, i+0.5, fill=False, edgecolor="red", linewidth=1.4)
ax.set_title("n_focal per doublet\ngreen=improved  red=regressed", fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.04, label="n_focal")
plt.tight_layout(); fig.savefig(OUT / "_heatmap.png", dpi=140, bbox_inches="tight"); plt.show()

In [ ]:
# Contact sheet — quick scan of the 'improved' doublets to confirm visually
improvers = df_s[df_s["n_treated"] > df_s["n_control"]].head(24)
if len(improvers):
    print(f"{len(improvers)} improved doublets — first {min(24, len(improvers))} shown")
    ncol = 4; nrow = (len(improvers) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3.2, nrow * 3.2))
    axes = axes.ravel() if hasattr(axes, 'ravel') else [axes]
    for ax, (_, r) in zip(axes, improvers.iterrows()):
        img_path = OUT / f"{r['benchmark_id']}.png"
        if img_path.exists():
            ax.imshow(plt.imread(img_path))
            ax.set_title(f"{r['benchmark_id']}: ctrl={r['n_control']} → cut={r['n_treated']}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[len(improvers):]: ax.set_visible(False)
    plt.tight_layout(); fig.savefig(OUT / "_improvers_contact_sheet.png", dpi=120, bbox_inches="tight"); plt.show()
else:
    print("No improvers to show.")